# 11 • OpenAI + MCP: ค้นพบเครื่องมือสถานะคลัง

เป้าหมาย: ให้ MCP server ประกาศ tools แล้วใช้ LLM เลือก tool ส่งผลกลับและสรุปภาษาไทย

**ข้อมูลสมมติสำหรับการอบรม:** ชื่อหน่วยงาน Lot ผู้ขาย ค่าที่วัด และ SOP ในตัวอย่างถูกสร้างขึ้นเพื่อเรียนรู้ AI ไม่ใช่ข้อมูลการผลิตหรือระเบียบจริงของกลุ่มศรีตรัง

บริบทงานอ้างอิงจาก [เว็บไซต์กลุ่มศรีตรัง](https://www.sritranggroup.com/th/home) ส่วนรูปแบบ notebook อ้างอิงจาก [ตัวอย่าง GitHub](https://github.com/biodatlab/cpf-genai-workshop/tree/main/tutorial_notebooks) และใช้ [OpenAI API documentation](https://developers.openai.com/api/docs) สำหรับ syntax

## เตรียมพร้อมก่อนเริ่ม
เพิ่ม Secret ชื่อ `OPENAI_API_KEY` ใน Colab ด้วยตนเอง และเปิด **Notebook access** สำหรับ notebook นี้ โค้ดจะอ่านผ่าน `userdata.get()` โดยไม่แสดงค่า key

รันเซลล์จากบนลงล่าง หลังติดตั้ง package และเตรียม key แล้ว ทดสอบตัวอย่างหลักใน Colab ด้วย API แล้ว แต่ผล LLM อาจเปลี่ยนได้ การผ่านตัวอย่างนี้ไม่ใช่การรับรองทุกคำถามหรือการใช้งานจริง

ใช้ `gpt-4.1-mini` สำหรับสร้างคำตอบ (ไม่เกิน 500 tokens ต่อ request) และ `text-embedding-3-small` สำหรับ Embedding โดย Agent Loop จำกัดไว้ 5 รอบ

In [1]:
%pip -q install openai "mcp>=1.28,<2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.6/234.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.8 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
MODEL = "gpt-4.1-mini"

## MCP server ที่เตรียมไว้
STDIO เปิด server เป็น process ภายใน Colab ไม่ต้องเปิด endpoint สาธารณะหรือเพิ่ม API key ใช้ SDK รุ่น v1 ที่กำหนดขอบเขตไว้

In [7]:
%%writefile flood_server.py
from mcp.server.fastmcp import FastMCP
from typing import Literal

mcp = FastMCP("สถานะคลังสำหรับการอบรม")

@mcp.tool()
def get_flood_level(factory: Literal["คลังยางแท่ง A", "คลังน้ำยาง B"]):
    """อ่านระดับน้ำสมมติของคลังยางแท่ง A หรือคลังน้ำยาง B สำหรับการอบรม"""
    readings = {"คลังยางแท่ง A": 55, "คลังน้ำยาง B": 10}
    if factory not in readings:
        return {"error": "ไม่พบหน่วยงานนี้", "source": "ข้อมูลสมมติสำหรับการอบรม"}
    return {"factory": factory, "water_cm": readings[factory], "source": "ข้อมูลสมมติสำหรับการอบรม", "live": False}

if __name__ == "__main__":
    mcp.run(transport="stdio")

Overwriting flood_server.py


In [8]:
import json
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server = StdioServerParameters(command=sys.executable, args=["flood_server.py"])

In [9]:
async def mcp_exercise():
    with open("mcp_stderr.log", "w", encoding="utf-8") as errlog:
        async with stdio_client(server, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                discovered = await session.list_tools()
                tools = [{"type": "function", "name": t.name,
                          "description": t.description, "parameters": t.inputSchema}
                         for t in discovered.tools]
                print("Tools ที่ MCP ประกาศ:", [t["name"] for t in tools])
                print("Schema จาก MCP:", json.dumps(tools, ensure_ascii=False))
                messages = [{"role": "user", "content": "ตรวจระดับน้ำสมมติที่คลังยางแท่ง A แล้วสรุปเป็นภาษาไทย"}]
                response = client.responses.create(model=MODEL, input=messages, tools=tools,
                                                   tool_choice="required", max_output_tokens=500)
                messages += response.output
                for call in response.output:
                    if call.type == "function_call":
                        print("Tool ที่เลือก:", call.name, call.arguments)
                        result = await session.call_tool(call.name, json.loads(call.arguments))
                        output = result.model_dump_json()
                        print("ผลจาก MCP:", output)
                        messages.append({"type": "function_call_output", "call_id": call.call_id,
                                         "output": output})
                final = client.responses.create(model=MODEL, input=messages,
                    instructions="ตอบเป็นภาษาไทยและระบุชัดเจนว่าข้อมูลนี้เป็นข้อมูลสมมติสำหรับการอบรม", max_output_tokens=500)
                print(final.output_text)

await mcp_exercise()

Tools ที่ MCP ประกาศ: ['get_flood_level']
Schema จาก MCP: [{"type": "function", "name": "get_flood_level", "description": "อ่านระดับน้ำสมมติของคลังยางแท่ง A หรือคลังน้ำยาง B สำหรับการอบรม", "parameters": {"properties": {"factory": {"enum": ["คลังยางแท่ง A", "คลังน้ำยาง B"], "title": "Factory", "type": "string"}}, "required": ["factory"], "title": "get_flood_levelArguments", "type": "object"}}]
Tool ที่เลือก: get_flood_level {"factory":"คลังยางแท่ง A"}
ผลจาก MCP: {"meta":null,"content":[{"type":"text","text":"{\n  \"factory\": \"คลังยางแท่ง A\",\n  \"water_cm\": 55,\n  \"source\": \"ข้อมูลสมมติสำหรับการอบรม\",\n  \"live\": false\n}","annotations":null,"meta":null}],"structuredContent":null,"isError":false}
ระดับน้ำที่คลังยางแท่ง A อยู่ที่ประมาณ 55 เซนติเมตร ข้อมูลนี้เป็นข้อมูลสมมติสำหรับการอบรมเท่านั้น ไม่ใช่ข้อมูลจริงค่ะ


## ลองทำด้วยตนเอง
1. เปลี่ยนคำถามเป็นคลังน้ำยาง B
2. เปรียบเทียบ schema ที่ค้นพบกับ notebook 10
3. อธิบายว่า MCP ทำให้ tool ใช้ซ้ำได้อย่างไร และใครเป็นผู้เลือก tool

ส่วนเสริมสำหรับวิทยากร: ใช้ HTTP MCP server ที่เข้าถึงได้ โดย native remote MCP ต้องมี endpoint ที่ OpenAI เข้าถึงได้